In [ ]:
import os, random
import numpy as np
import torch, evaluatew
from datasets import load_dataset, Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                        TrainingArguments, Trainer)

# !mamba update -c conda-forge accelerate
# 

c:\Users\uqpua\miniforge3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import pandas as pd
from datasets import Dataset

df = pd.read_csv("train.csv")
dataset = Dataset.from_pandas(df)


ds = Dataset.from_dict({"text": texts, "label": labels})

In [3]:
# TODO: implement
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

# TODO: implement
def build_datasets(tokenizer, max_length: int = 128): # train_path: str, val_path: str
    """
    return train_dataset, val_dataset
    """
    def tokenize_function(batch):
        return tokenizer(
            batch["text"],
            truncation=True,
            padding="max_length",
            max_length=max_length
        )

    ds = load_dataset("imdb")
    # ds = load_dataset(
    #     "csv",
    #     data_files={"train": train_path, "validation": val_path},
    # )
    
    # ds["train"].select(range(100))
    # ds["train"].shuffle(seed=42).select(range(train_size))
    train_size = 100
    train_ds = ds["train"].shuffle(seed=42).select(range(train_size))
    val_ds = ds["test"].shuffle(seed=42).select(range(train_size))
    
    train_tok = train_ds.map(tokenize_function, batched=True)
    val_tok = val_ds.map(tokenize_function, batched=True)

    # label -> labels 로 통일
    train_tok = train_tok.rename_column("label", "labels")
    val_tok   = val_tok.rename_column("label", "labels")

    train_tok.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
    val_tok.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

    return train_tok, val_tok

# TODO: implement
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")
def compute_metrics(eval_pred):
    """
    return dict with 'accuracy' (and/or f1)
    """
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    # Accuracy 계산
    # acc = accuracy.compute(predictions=predictions, references=labels)
    
    # F1 Score 계산
    # 이진 분류(IMDB 등)라면 기본값으로 충분하지만, 
    # 다중 분류라면 average="weighted" 또는 "macro"를 설정해야 합니다.
    # f1 = f1.compute(predictions=predictions, references=labels, average="binary")
    
    # return {
    #     "accuracy": acc["accuracy"],
    #     "f1": f1["f1"]
    # }
    
    return accuracy.compute(predictions=preds, references=labels)

# TODO: implement
def train(model_name: str ="distilbert-base-uncased", 
          output_dir: str ="./results", 
          num_labels = 2):
    """
    1) load tokenizer/model
    2) build datasets
    3) train
    4) evaluate and save best model
    """
    set_seed(42)
    device = torch.device("cpu")

    # 1)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, 
                                                               num_labels=num_labels) 
    model.to(device)

    # 2)
    train_dataset, val_dataset = build_datasets(tokenizer = tokenizer,
                                                max_length = 128) # train_csv, val_csv,

    # 3)
    training_args = TrainingArguments(
        output_dir=output_dir,
        eval_strategy="epoch",  # steps, epoch
        learning_rate=2e-5,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=8,
        num_train_epochs=1,
        save_strategy="no",
        logging_strategy="steps",
        logging_steps=10,
        # report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        processing_class=tokenizer,
        compute_metrics=compute_metrics
    )
    trainer.train()

    # 4)
    metrics = trainer.evaluate()
    print(metrics)

    # trainer.save_model(output_dir)

    return metrics    

# if __name__ == "__main__":
#     train()

In [4]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
# 2)
train_dataset, val_dataset = build_datasets(tokenizer = tokenizer,
                                            max_length = 128) # train_csv, val_csv,


In [8]:
train_dataset[0]

{'labels': tensor(1),
 'input_ids': tensor([  101,  2045,  2003,  2053,  7189,  2012,  2035,  2090,  3481,  3771,
          1998,  6337,  2099,  2021,  1996,  2755,  2008,  2119,  2024,  2610,
          2186,  2055,  6355,  6997,  1012,  6337,  2099,  3504, 15594,  2100,
          1010,  3481,  3771,  3504,  4438,  1012,  6337,  2099, 14811,  2024,
          3243,  3722,  1012,  3481,  3771,  1005,  1055,  5436,  2024,  2521,
          2062,  8552,  1012,  1012,  1012,  3481,  3771,  3504,  2062,  2066,
          3539,  8343,  1010,  2065,  2057,  2031,  2000,  3962, 12319,  1012,
          1012,  1012,  1996,  2364,  2839,  2003,  5410,  1998,  6881,  2080,
          1010,  2021,  2031,  1000, 17936,  6767,  7054,  3401,  1000,  1012,
          2111,  2066,  2000, 12826,  1010,  2000,  3648,  1010,  2000, 16157,
          1012,  2129,  2055,  2074,  9107,  1029,  6057,  2518,  2205,  1010,
          2111,  3015,  3481,  3771,  3504,  2137,  2021,  1010,  2006,  1996,
          2060,  

In [9]:
train_dataset

Dataset({
    features: ['text', 'labels', 'input_ids', 'attention_mask'],
    num_rows: 100
})

In [14]:
train_dataset[5]

{'labels': tensor(1),
 'input_ids': tensor([  101,  2096,  2023,  3185,  1005,  1055,  2806,  3475,  1005,  1056,
          2004,  2104,  9153,  3064,  1998, 12689,  2004,  1037,  2614,  2544,
          2763,  2052,  2031,  2042,  1010,  2023,  2003,  2145,  1037,  2200,
          2204,  2143,  1012,  1999,  2755,  1010,  2009,  2001,  2464,  2004,
          2019,  6581,  2143,  1999,  2049,  2154,  1010,  2004,  2009,  2001,
          4222,  2005,  1996,  2034,  2190,  3861,  7436,  1006,  3974,  2000,
          4777,  1007,  1012,  1045,  2145,  5136,  4777,  2000,  2022,  1037,
          6020,  2143,  1010,  2021,  2023,  2028,  2003,  6581,  2750,  1037,
          2210,  2978,  1997,  2058, 18908,  2075,  2011,  1996,  2599,  1010,
         16243,  5553,  5582,  2015,  1012,  1026,  7987,  1013,  1028,  1026,
          7987,  1013,  1028,  5553,  5582,  2015,  2003,  1037,  2236,  2013,
          1039,  9057,  2923,  3607,  2040,  2003,  2542,  2041,  2010,  2345,
          2420,  